# 3 · Who reacts? Heterogeneous effects and uplift

The ATE hides heterogeneity. Meta-learners estimate the conditional effect τ(x) = E[Y(1) − Y(0) | X = x]; we rank customers by τ̂(x) and evaluate the ranking on a held-out set with Qini curves.
Propensity is known (0.5) because the data is randomized. Qini coefficients are normalized by the perfect ranking (treated responders first, control responders last).

**Segments** use the best model's τ̂(x) and the baseline P(Y | no e-mail): *Persuadables* τ̂ > δ, *Sleeping Dogs* τ̂ < −δ; otherwise *Sure Things* (high baseline) or *Lost Causes* (low baseline), with δ = ATE / 2. The segment table checks each group's uplift on held-out data.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from uplift_causal.data import load_hillstrom
from uplift_causal.evaluate_uplift import (
    budget_table, diff_means, qini_ci, qini_curve, segment_table, segments
)
from uplift_causal.learners import baseline, fit_predict
from uplift_causal.plots import qini_plot, quadrant_plot, save

In [2]:
df = load_hillstrom()
train, test = train_test_split(df, test_size=0.4, stratify=df.t * 2 + df.y, random_state=0)
X_train, X_test = train.drop(columns=["t", "y"]), test.drop(columns=["t", "y"])
models, tau = fit_predict(X_train, train.t, train.y, X_test)

In [ ]:
qini = pd.DataFrame(
    {k: qini_ci(test.y, test.t, v) for k, v in tau.items()}, index=["qini", "lo", "hi"]
).T
best = qini.qini.idxmax()
qini.round(4)

In [4]:
save(qini_plot({k: qini_curve(test.y, test.t, v) for k, v in tau.items()}), "qini")

In [ ]:
ate = diff_means(train.y, train.t)[0]
threshold, delta = train.y[train.t == 0].mean(), ate / 2
mu0 = baseline(models["T-Learner"], X_test)
seg = segments(mu0, tau[best], threshold, delta)
segment_table(test.y, test.t, seg).round(4)

In [ ]:
save(quadrant_plot(mu0, tau[best], seg, threshold, delta), "segments", height=600)

In [ ]:
budget = budget_table(test.y, test.t, tau[best])
row = budget.iloc[2]
print(
    f"{best}: targeting the top {row.cost_share:.0%} captures {row.effect_share:.0%} "
    f"(95% CI {row.lo:.0%}-{row.hi:.0%}) of the incremental visits."
)
budget.round(3)